In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [13]:
import os
import pickle as pkl

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data

SAVE = False  # whether to save the outputs of this notebook

if SAVE:
    # Replace this with the path to directory where you would like results to be saved
    OUTPUT_PATH = "../results/"
    os.makedirs(OUTPUT_PATH, exist_ok=True)  # Create path if it doesn't exist

This notebook reproduces the results from a study on chronic hypersensitivity pneumonitis. See publication here: https://pmc.ncbi.nlm.nih.gov/articles/PMC7667907/pdf/rccm.202001-0134OC.pdf

In [2]:
counts = pd.read_csv(f'/Users/anugulati/projects/chp-project/data/GSE150910_gene-level_count_file.csv')

In [3]:
counts.shape

(18838, 289)

In [6]:
counts.head()

,symbol,chp_26,chp_31,chp_34,chp_38,chp_1,chp_3,chp_4,chp_11,chp_5,...,chp_106,chp_108,chp_109,chp_110,chp_111,chp_112,chp_113,chp_114,ipf_1125,ipf_1130
0,TSPAN6,1361,993,351,613,841,565,1093,6239,1714,...,466,638,334,349,778,872,693,465,639,944
1,TNMD,5,13,0,0,0,6,3,4,0,...,0,0,0,0,0,0,0,0,0,0
2,DPM1,1929,2775,1894,2007,1436,1923,1956,4518,2411,...,936,1162,1043,689,1006,870,1701,860,1387,2150
3,SCYL3,176,216,208,218,162,137,242,490,259,...,57,35,122,42,73,68,145,21,146,204
4,C1orf112,93,143,97,148,98,128,102,188,119,...,50,60,72,21,74,48,131,72,69,158


In [8]:
counts.columns

Index(['symbol', 'chp_26', 'chp_31', 'chp_34', 'chp_38', 'chp_1', 'chp_3',
       'chp_4', 'chp_11', 'chp_5',
       ...
       'chp_106', 'chp_108', 'chp_109', 'chp_110', 'chp_111', 'chp_112',
       'chp_113', 'chp_114', 'ipf_1125', 'ipf_1130'],
      dtype='object', length=289)

In [4]:
from collections import defaultdict

In [11]:
groups = defaultdict(int)

for sample in counts.columns:
    if sample == 'symbol': continue
    group = sample.split('_')[0]
    # print(group)
    groups[group] += 1


In [12]:
groups

defaultdict(int, {'chp': 82, 'ipf': 103, 'control': 103})

In [ ]:
# break matrix into two matrices for chp + control and ipf + control

In [14]:
chp_sample_ids = []; ipf_sample_ids = []; control_sample_ids = []

for sample in counts.columns:
    group = sample.split('_')[0]
    if group == 'chp':
        chp_sample_ids.append(sample)
    elif group == 'ipf': 
        ipf_sample_ids.append(sample)
    elif group == 'control':
        control_sample_ids.append(sample)

In [15]:
len(chp_sample_ids)

82

In [16]:
len(ipf_sample_ids)

103

In [17]:
len(control_sample_ids)

103

In [26]:
counts = counts.set_index('symbol')

Prepare counts matrix for DESeq2:

In [27]:
counts_chp_control = counts[chp_sample_ids + control_sample_ids]

In [30]:
counts_chp_control = counts_chp_control.T # transform into samples by genes

In [31]:
counts_chp_control

symbol,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,...,SPEGNB,AL353579.1,AC053503.7,AC012488.2,AC242842.3,AC055839.2,AC242843.1,SPDYE17,AC022137.3,AC007244.1
chp_26,1361,5,1929,176,93,1249,8098,1477,772,515,...,47,11,0,0,14,759,27,3,71,0
chp_31,993,13,2775,216,143,1583,8392,2292,660,504,...,19,16,8,0,43,354,37,1,51,12
chp_34,351,0,1894,208,97,2767,3409,1547,967,468,...,0,11,4,0,86,202,71,3,60,0
chp_38,613,0,2007,218,148,2968,5155,2286,980,707,...,18,0,0,0,75,358,84,2,106,21
chp_1,841,0,1436,162,98,2444,3242,1737,657,552,...,13,23,0,0,80,756,29,3,26,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
control_1102,577,0,2045,213,87,5423,2294,1696,302,732,...,55,0,5,0,61,391,53,1,78,0
control_1109,655,0,2060,158,83,1711,2424,2308,600,430,...,55,7,19,0,47,343,36,0,39,0
control_1111,1028,5,2841,216,192,1891,6073,3587,629,502,...,83,34,13,0,23,327,15,1,84,0
control_1112,780,0,2304,257,143,1725,7295,2400,574,682,...,80,37,0,0,41,464,25,1,67,0


In [ ]:
metadata_chp_control = 